# Experiment runner: E1 benchmark and E3 noisy neighbours
Runs the main experiments of the CS545 GAT project on a Colab GPU (T4).
The project code (`src/`, `scripts/`, `tests/`) is cloned from <https://github.com/YashDolorase/cs545-gat>.
The results are the CSV files that `GAT_Project.ipynb` reads.

| Experiment | Setup | Runs |
|---|---|---|
| E1 | MLP, GCN, GraphSAGE, GAT on Cora and CiteSeer | 4 models × 2 datasets × 10 seeds |
| E3 | Same four models on Cora with 0 / 10 / 25 / 50 / 100 % random edges added | 5 noise levels × 4 models × 10 seeds |

## Setup

In [1]:
import os, torch

# Clone the project repository and install PyTorch Geometric (Colab ships PyTorch but not PyG)
PROJECT = "/content/project"
if not os.path.exists(PROJECT):
    !git clone -q https://github.com/YashDolorase/cs545-gat.git {PROJECT}
os.chdir(PROJECT)
!pip -q install torch_geometric

import torch_geometric
print(f"torch {torch.__version__} | PyG {torch_geometric.__version__} | "
      f"device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


torch 2.14.1+cu130 | PyG 2.8.1 | device: Tesla T4


In [2]:
# Results directory. The runner skips any (dataset, model, seed) already in its CSV, so it starts empty
# and every run trains from scratch. Google Drive is used when available, so the results survive a disconnect.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT = "/content/drive/MyDrive/CS545_GAT/fresh_run"
except Exception:
    OUT = "/content/fresh_run"
os.makedirs(OUT, exist_ok=True)
print("results ->", OUT, "| existing files:", os.listdir(OUT) or "none")

results -> /content/fresh_run | existing files: none


## Sanity tests
* The edge-wise softmax sums to 1 over every node's neighbourhood.
* The from-scratch GAT layer (`src/gat_layer.py`) gives the same outputs as PyG's `GATConv` (concatenated and averaged heads).
* Random-edge injection adds the requested number of edges: new, undirected, with no duplicates or self-loops.
* With the attention vectors set to zero, attention is uniform: the measured entropy equals the uniform entropy.

In [3]:
os.chdir(PROJECT)  # Colab resets the working directory after a reconnect
!python -m pytest tests -v -W ignore

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/project
plugins: typeguard-4.6.0, anyio-4.16.0, platformdirs-4.14.3, langsmith-0.14.17
collected 4 items                                                              

tests/test_sanity.py::test_segment_softmax_sums_to_one PASSED            [ 25%]
tests/test_sanity.py::test_gat_layer_matches_pyg PASSED                  [ 50%]
tests/test_sanity.py::test_noise_edges PASSED                            [ 75%]
tests/test_sanity.py::test_attention_stats_uniform PASSED                [100%]

============================== 4 passed in 7.56s ===============================


## E1: Benchmark
Each model uses the hyperparameters from its original paper and is trained full-batch with early stopping on validation accuracy.
Test accuracy comes from the standard public split. Ten seeds give the run-to-run spread that GAT's margin is compared against.
Results → `e1_benchmark.csv`.

In [4]:
os.chdir(PROJECT)
!python -W ignore scripts/run_experiments.py --exp e1 --seeds 10 --out {OUT}

device: cuda | torch 2.14.1+cu130
Processing...
Done!
[E1] Cora      MLP    seed 0: test 0.585 (best ep 154, 2.0 ms/ep)
[E1] Cora      GCN    seed 0: test 0.817 (best ep 260, 3.7 ms/ep)
[E1] Cora      SAGE   seed 0: test 0.814 (best ep 118, 4.2 ms/ep)
[E1] Cora      GAT    seed 0: test 0.830 (best ep 210, 7.6 ms/ep)
[E1] Cora      MLP    seed 1: test 0.587 (best ep 167, 2.1 ms/ep)
[E1] Cora      GCN    seed 1: test 0.814 (best ep 298, 3.8 ms/ep)
[E1] Cora      SAGE   seed 1: test 0.817 (best ep 70, 3.7 ms/ep)
[E1] Cora      GAT    seed 1: test 0.818 (best ep 21, 6.5 ms/ep)
[E1] Cora      MLP    seed 2: test 0.577 (best ep 145, 1.9 ms/ep)
[E1] Cora      GCN    seed 2: test 0.825 (best ep 164, 4.2 ms/ep)
[E1] Cora      SAGE   seed 2: test 0.814 (best ep 115, 4.1 ms/ep)
[E1] Cora      GAT    seed 2: test 0.818 (best ep 238, 6.4 ms/ep)
[E1] Cora      MLP    seed 3: test 0.591 (best ep 112, 1.9 ms/ep)
[E1] Cora      GCN    seed 3: test 0.825 (best ep 162, 3.5 ms/ep)
[E1] Cora      SAGE   se

## E3: Noisy neighbours
Random undirected edges between unconnected node pairs are added to Cora, mostly joining different classes.
Accuracy is recorded at each noise level, along with the attention GAT gives injected edges compared with original ones.
Results → `e3_noise.csv`.

In [5]:
os.chdir(PROJECT)
!python -W ignore scripts/run_experiments.py --exp e3 --seeds 10 --out {OUT}

device: cuda | torch 2.14.1+cu130
[E3] Cora noise 0.00 MLP    seed 0: test 0.585
[E3] Cora noise 0.00 GCN    seed 0: test 0.817
[E3] Cora noise 0.00 SAGE   seed 0: test 0.814
[E3] Cora noise 0.00 GAT    seed 0: test 0.830
[E3] Cora noise 0.00 MLP    seed 1: test 0.587
[E3] Cora noise 0.00 GCN    seed 1: test 0.814
[E3] Cora noise 0.00 SAGE   seed 1: test 0.817
[E3] Cora noise 0.00 GAT    seed 1: test 0.818
[E3] Cora noise 0.00 MLP    seed 2: test 0.577
[E3] Cora noise 0.00 GCN    seed 2: test 0.825
[E3] Cora noise 0.00 SAGE   seed 2: test 0.814
[E3] Cora noise 0.00 GAT    seed 2: test 0.818
[E3] Cora noise 0.00 MLP    seed 3: test 0.591
[E3] Cora noise 0.00 GCN    seed 3: test 0.825
[E3] Cora noise 0.00 SAGE   seed 3: test 0.819
[E3] Cora noise 0.00 GAT    seed 3: test 0.829
[E3] Cora noise 0.00 MLP    seed 4: test 0.588
[E3] Cora noise 0.00 GCN    seed 4: test 0.820
[E3] Cora noise 0.00 SAGE   seed 4: test 0.821
[E3] Cora noise 0.00 GAT    seed 4: test 0.826
[E3] Cora noise 0.00 MLP  

## Results summary

In [6]:
import pandas as pd

# E1: test accuracy (%) as mean ± std over seeds
e1 = pd.read_csv(f"{OUT}/e1_benchmark.csv")
s = e1.groupby(["dataset", "model"])["test_acc"].agg(["mean", "std", "count"])
s["test acc (%)"] = (100 * s["mean"]).round(1).astype(str) + " ± " + (100 * s["std"]).round(1).astype(str)
display(s[["test acc (%)", "count"]].unstack("dataset"))

# E3: mean test accuracy (%) per noise level (rows) and model (columns)
e3 = pd.read_csv(f"{OUT}/e3_noise.csv")
display((100 * e3.pivot_table(index="noise", columns="model", values="test_acc", aggfunc="mean")).round(1))

test acc (%)                count     
dataset     CiteSeer        Cora CiteSeer Cora
model                                         
GAT       71.5 ± 0.9  82.3 ± 0.7       10   10
GCN       71.0 ± 0.4  82.1 ± 0.5       10   10
MLP       57.3 ± 1.1  59.0 ± 0.9       10   10
SAGE      70.2 ± 0.7  81.3 ± 0.6       10   10

model,GAT,GCN,MLP,SAGE
noise,,,,
0.00,82.3,82.1,59.0,81.3
0.10,80.4,80.0,59.0,79.6
0.25,77.4,77.9,59.0,77.4
0.50,73.0,73.9,59.0,74.2
1.00,69.0,68.2,59.0,71.6


In [7]:
# Package the results directory for download
!cd {os.path.dirname(OUT)} && zip -qr /content/fresh_run.zip {os.path.basename(OUT)}
from google.colab import files
files.download("/content/fresh_run.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>